# 10주차 LSTM 으로 영화 리뷰 감성 분류 — 3교시 실습 〔빈칸본〕

**목표**: 한국어 영화 리뷰(NSMC)를 ① **숫자로 바꾸고**(어휘사전·인덱스·패딩) ② **임베딩 → LSTM → Linear** 모델로
긍정/부정을 분류한 뒤 ③ **학습 곡선·혼동행렬·내 문장 예측**으로 결과를 확인한다.

```
  1단계 데이터와 어휘사전            ← 1교시 "단어에 번호 붙이기"
  2단계 문장을 숫자로 (인덱스·패딩)   ← 1교시 "길이를 맞춘다"
  3단계 LSTM 모델 만들기              ← 1교시 임베딩 + 2교시 LSTM
  4단계 학습 + 가장 좋은 모델 저장     ← 9주차와 같은 학습 함수
  5단계 평가 (곡선 · 혼동행렬 · 틀린 문장)
  6단계 내 문장 넣어 보기
  도전  어텐션 — 모델이 어느 단어를 봤나 (선택)   ← 2교시 "어텐션의 아이디어"
```

### 빈칸 채우는 법

- 코드 안의 **`____`** 를 지우고 알맞은 코드를 적습니다. `____` 한 줄 = 코드 한 줄입니다.
- 빈칸 바로 위의 **`# 빈칸 ⓝ` 힌트**를 먼저 읽으세요. 대부분 정답 형태가 힌트 안에 있습니다.
- 셀을 실행한 뒤 **아래 "기대 출력"과 비교**합니다. 비슷하면 통과입니다.
- 빈칸은 모두 **10곳 · 12줄**입니다.

> 같은 폴더에 **`preprocess_text.py`** 가 있어야 합니다. 이 파일이 토큰화·어휘사전·패딩을 대신 해 줍니다.
> 데이터(`data/nsmc_subset/`)가 없으면 셀 1 이 원본(약 19MB)을 내려받아 만듭니다.

---

## 0. 준비 — 라이브러리와 장치

In [ ]:
# 셀 0 — 준비  (빈칸 없음)
import os, json, time, random
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from preprocess_text import load_nsmc, make_subset, tokenize, build_vocab, encode, encode_all, decode, save_vocab

plt.rcParams["font.family"] = "Malgun Gothic"     # 그래프 한글
plt.rcParams["axes.unicode_minus"] = False

def set_seed(s=42):                               # 어느 PC 에서나 같은 결과가 나오게
    random.seed(s); torch.manual_seed(s)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRAIN, EPOCHS = 20000, 5                        # CPU 로도 몇 분이면 끝나므로 GPU·CPU 같은 크기  ★ 바꾸지 마세요
MAX_LEN = 40                                      # 문장 최대 길이 (단어 40개)

print("장치 :", device, f"| 훈련 문장 {N_TRAIN:,}개 / {EPOCHS} epoch")
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

---

## 1단계 — 데이터와 어휘사전

**NSMC** 는 네이버 영화 리뷰 데이터입니다. 리뷰 한 줄과 정답(**0 = 부정, 1 = 긍정**)이 짝지어 있습니다.

**어휘사전**은 *"단어 → 번호"* 표입니다. 1교시 판서처럼 **훈련 문장으로만** 만듭니다.
검증 문장에서 처음 보는 단어는 `<unk>`(모르는 단어, 1번)로 바뀝니다.

In [ ]:
# 셀 1 — 데이터 불러오기 + 어휘사전 만들기
DATA = "data/nsmc_subset"
if not os.path.exists(f"{DATA}/train.tsv"):      # 없으면 원본을 내려받아 만든다
    make_subset(DATA)

train_texts, train_labels, val_texts, val_labels = load_nsmc(DATA)
train_texts, train_labels = train_texts[:N_TRAIN], train_labels[:N_TRAIN]
print(f"훈련 {len(train_texts):,}문장 / 검증 {len(val_texts):,}문장")
for t, l in zip(train_texts[:3], train_labels[:3]):
    print(f"  [{'긍정' if l else '부정'}] {t}")

# 빈칸 ① (1줄) — 힌트 : 훈련 문장으로만 어휘사전을 만든다. 크기 상한 2만.
#                      build_vocab(train_texts, max_size=20000)
vocab = ____
VOCAB_SIZE = len(vocab)
print(f"\n어휘사전 크기 : {VOCAB_SIZE:,}")
print("앞 8개 :", list(vocab.items())[:8])

**기대 출력** (교수 PC · CPU)
```
훈련 20,000문장 / 검증 4,000문장
  [부정] 미국식유머??나도 미국식유머좋아하는데 이건진짜 재미없다..보다가 끔
  [부정] 언어를 통해 전염되는 좀비병? 설정이 너무 이상해
  [긍정] 좋은 영화인데 개봉을 너무 짧게해서 아쉬웠다.

어휘사전 크기 : 14,012
앞 8개 : [('<pad>', 0), ('<unk>', 1), ('.', 2), ('ㅋ', 3), ('!', 4), (',', 5), ('영화', 6), ('?', 7)]
```

> **확인**: 0번은 `<pad>`(빈칸 채우기), 1번은 `<unk>`(모르는 단어)입니다. 그다음부터는 **자주 나온 단어 순서**입니다.

> **막히면**: `ModuleNotFoundError: preprocess_text` → `preprocess_text.py` 가 노트북과 **같은 폴더**에 있는지 확인하세요.

---

## 2단계 — 문장을 숫자로 (인덱스 · 패딩)

```
  "이 영화 정말 재미없다"
     → 토큰  ["이", "영화", "정말", "재미없다"]
     → 번호  [ 12,   5,     30,     842 ]
     → 패딩  [ 12, 5, 30, 842, 0, 0, ..., 0 ]     ← 길이 40 으로 맞춘다 (0 = <pad>)
```

In [ ]:
# 셀 2 — 한 문장 바꿔 보기 + 전체를 텐서로
s = "이 영화 정말 재미없다"
print("토큰 :", tokenize(s))
# 빈칸 ② (1줄) — 힌트 : 문장 s 를 길이 MAX_LEN 의 번호 리스트로.  encode(s, vocab, MAX_LEN)
ids = ____
print("번호 :", ids[:10], "...")
print("되돌리기 :", decode(ids, vocab))

X  = torch.tensor(encode_all(train_texts, vocab, MAX_LEN));  y  = torch.tensor(train_labels)
Xv = torch.tensor(encode_all(val_texts,   vocab, MAX_LEN));  yv = torch.tensor(val_labels)
train_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=256)

print("\n훈련 텐서 :", tuple(X.shape), "| 검증 텐서 :", tuple(Xv.shape))
real = (X != 0).sum(dim=1).float()
print(f"실제 단어 수 평균 {real.mean():.1f}개 → 40칸 중 약 {(1 - real.mean() / MAX_LEN) * 100:.0f}% 가 패딩")

**기대 출력**
```
토큰 : ['이', '영화', '정말', '재미없다']
번호 : [18, 6, 10, 155, 0, 0, 0, 0, 0, 0] ...
되돌리기 : ['이', '영화', '정말', '재미없다']

훈련 텐서 : (20000, 40) | 검증 텐서 : (4000, 40)
실제 단어 수 평균 10.9개 → 40칸 중 약 73% 가 패딩
```

> **관찰 ★**: 리뷰는 대부분 짧아서 **40칸 중 상당 부분이 패딩(0)** 입니다.
> 그래서 3단계 모델은 **패딩까지 다 읽은 마지막 값이 아니라, 마지막 '실제 단어' 위치의 값**을 꺼내 씁니다.

---

## 3단계 — LSTM 모델 만들기

```
  번호 (B, 40)
   → nn.Embedding  : 번호마다 128개 숫자(임베딩 벡터)     → (B, 40, 128)    ← 1교시
   → nn.LSTM       : 앞에서부터 읽으며 요약을 넘긴다      → (B, 40, 128)    ← 2교시
   → 마지막 실제 단어 위치의 요약만 꺼낸다                → (B, 128)
   → Dropout → nn.Linear → 부정/긍정 점수                 → (B, 2)
```

In [ ]:
# 셀 3 — 모델 정의
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed=128, hidden=128):
        super().__init__()
        # 빈칸 ③ (1줄) — 힌트 : 번호 → 128개 숫자.  0번(<pad>)은 학습에서 뺀다
        #                      nn.Embedding(vocab_size, embed, padding_idx=0)
        self.emb = ____
        # 빈칸 ④ (1줄) — 힌트 : 입력 128 → 요약 128.  batch_first=True 를 꼭 넣는다
        #                      nn.LSTM(embed, hidden, batch_first=True)
        self.lstm = ____
        self.drop = nn.Dropout(0.3)
        # 빈칸 ⑤ (1줄) — 힌트 : 요약 128개 → 점수 2개 (부정 · 긍정)
        self.fc = nn.Linear(____, ____)

    def forward(self, x):
        e = self.emb(x)                                    # (B, 40) → (B, 40, 128)
        out, (h, c) = self.lstm(e)                         # out : 모든 시점의 요약 (B, 40, 128)
        length = (x != 0).sum(dim=1).clamp(min=1)          # 패딩을 뺀 실제 길이
        last = out[torch.arange(len(x)), length - 1]       # 마지막 '실제 단어' 시점의 요약 (B, 128)
        return self.fc(self.drop(last))

set_seed()
model = LSTMClassifier(VOCAB_SIZE).to(device)

xb, yb = next(iter(train_loader))
print("입력 :", tuple(xb.shape), "→ 출력 :", tuple(model(xb.to(device)).shape))
n_all = sum(p.numel() for p in model.parameters())
n_emb = model.emb.weight.numel()
print(f"파라미터 {n_all:,}개  (그중 임베딩 {n_emb:,}개 = {n_emb / n_all * 100:.0f}%)")

**기대 출력**
```
입력 : (64, 40) → 출력 : (64, 2)
파라미터 1,925,890개  (그중 임베딩 1,793,536개 = 93%)
```

> **관찰 ★**: 파라미터의 대부분이 **임베딩 표**에 있습니다. 단어 2만 개 × 숫자 128개이기 때문입니다.

> **막히면**
> | 증상 | 원인 |
> |---|---|
> | `too many values to unpack` | LSTM 은 `out, (h, c)` 를 돌려준다. 빈칸 ④ 가 `nn.RNN` 이면 이 오류 |
> | 출력이 `(64, 128)` | 빈칸 ⑤ 를 비웠다 |
> | `index out of range in self` | 빈칸 ③ 의 첫 숫자가 어휘사전 크기(`vocab_size`)가 아니다 |

---

## 4단계 — 학습 + 가장 좋은 모델 저장

학습 함수는 **9주차 3교시와 같은 모양**입니다. 데이터와 모델만 바뀌었습니다.

In [ ]:
# 셀 4 — 평가 함수와 학습 함수
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()                                  # 평가 모드 (Dropout 끔)
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        total_loss += loss_fn(out, yb).item() * len(yb)
        correct += (out.argmax(1) == yb).sum().item()
        n += len(yb)
    return total_loss / n, correct / n


def fit(model, epochs, lr, save_path):
    set_seed()
    # 빈칸 ⑥ (1줄) — 힌트 : 9주차와 같다.  torch.optim.Adam(model.parameters(), lr=lr)
    optimizer = ____
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_acc, t0 = 0.0, time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        run, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)                       # ① 예측
            loss = loss_fn(out, yb)               # ② 손실
            # 빈칸 ⑦ (3줄) — 힌트 : ③ 기울기 지우기 → ④ 역전파 → ⑤ 갱신
            ____
            ____
            ____
            run += loss.item() * len(yb); n += len(yb)

        val_loss, val_acc = evaluate(model, val_loader)
        hist["train_loss"].append(run / n)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)

        mark = ""
        if val_acc > best_acc:                    # 지금까지 중 최고면 저장
            best_acc = val_acc
            # 빈칸 ⑧ (1줄) — 힌트 : 가중치만 파일로.  torch.save(model.state_dict(), save_path)
            ____
            mark = "  ← 최고, 저장"
        print(f"epoch {epoch} | 훈련 손실 {run/n:.4f} | 검증 손실 {val_loss:.4f} "
              f"| 검증 정확도 {val_acc*100:5.2f}%{mark}")

    hist["best_acc"], hist["time"] = best_acc, time.time() - t0
    print(f"\n최고 검증 정확도 {best_acc*100:.2f}%  |  학습 시간 {hist['time']:.1f}초  |  저장 : {save_path}")
    return hist


def plot_history(hist, title):
    ep = range(1, len(hist["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(ep, hist["train_loss"], "o-", label="훈련 손실")
    ax[0].plot(ep, hist["val_loss"], "o-", label="검증 손실")
    ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title(f"{title} — 손실")
    ax[1].plot(ep, [a * 100 for a in hist["val_acc"]], "o-", color="tab:green")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].set_title(f"{title} — 검증 정확도")
    plt.tight_layout(); plt.show()

import inspect                                    # 빈칸을 비워 두었는지 검사
if "____" in inspect.getsource(fit):
    print("⚠️ 빈칸 ⑥ ⑦ ⑧ 중 아직 채우지 않은 곳이 있습니다")
else:
    print("함수 준비 완료 : evaluate / fit / plot_history")

In [ ]:
# 셀 5 — LSTM 학습  (빈칸 없음)
lstm_hist = fit(model, epochs=EPOCHS, lr=1e-3, save_path="models/lstm_sentiment.pt")
plot_history(lstm_hist, "LSTM")

**기대 출력** (교수 PC CPU 실측 · 다른 작업과 동시 실행 · 학습 301.6초)
```
epoch 1 | 훈련 손실 0.6186 | 검증 손실 0.5502 | 검증 정확도 71.43%  ← 최고, 저장
epoch 2 | 훈련 손실 0.4901 | 검증 손실 0.4948 | 검증 정확도 75.38%  ← 최고, 저장
epoch 3 | 훈련 손실 0.3897 | 검증 손실 0.4841 | 검증 정확도 75.67%  ← 최고, 저장
epoch 4 | 훈련 손실 0.3062 | 검증 손실 0.5106 | 검증 정확도 76.55%  ← 최고, 저장
epoch 5 | 훈련 손실 0.2345 | 검증 손실 0.5762 | 검증 정확도 76.50%

최고 검증 정확도 76.55%  |  학습 시간 301.6초  |  저장 : models/lstm_sentiment.pt
```

> **관찰 ★ — 9주차 1교시의 과적합을 찾아보세요.** 훈련 손실은 0.6186 → 0.2345 로 계속 내려가는데, **검증 손실은 epoch 3 (0.4841) 이 가장 낮고 그 뒤로 0.5762 까지 다시 올라갑니다** — 과적합 신호입니다. 검증 정확도는 epoch 4 의 76.55% 가 최고라 그때 저장됐습니다. epoch 을 더 늘려도 좋아지지 않습니다.

> **막히면**: 정확도가 50% 근처에서 움직이지 않으면 빈칸 ⑦ 의 순서(`zero_grad → backward → step`)를 확인하세요.
> 긍정·부정이 반반이라 **아무렇게나 찍으면 50%** 입니다.

---

## 5단계 — 평가: 혼동행렬과 틀린 문장

In [ ]:
# 셀 6 — 최고 모델 불러와 혼동행렬 · 틀린 문장 보기
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# 빈칸 ⑨ (1줄) — 힌트 : 셀 5 에서 저장한 최고 가중치를 다시 넣는다.
#                      model.load_state_dict(torch.load("models/lstm_sentiment.pt"))
____

@torch.no_grad()
def predict_all(model):
    model.eval(); ps = []
    for xb, _ in val_loader:
        ps += model(xb.to(device)).argmax(1).cpu().tolist()
    return ps

preds = predict_all(model)
NAMES = ["부정", "긍정"]
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ConfusionMatrixDisplay(confusion_matrix(val_labels, preds), display_labels=NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_xlabel("모델의 예측"); ax.set_ylabel("정답"); ax.set_title("LSTM — 검증 문장")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=120); plt.show()
print(classification_report(val_labels, preds, target_names=NAMES, digits=3))

print("[틀린 문장 5개]")
wrong = [i for i, (p, t) in enumerate(zip(preds, val_labels)) if p != t]
for i in wrong[:5]:
    print(f"  정답 {NAMES[val_labels[i]]} / 예측 {NAMES[preds[i]]} | {val_texts[i]}")

**기대 출력** (혼동행렬 그림 아래)
```
              precision    recall  f1-score   support

          부정      0.731     0.840     0.782      2000
          긍정      0.812     0.692     0.747      2000

    accuracy                          0.765      4000
   macro avg      0.771     0.766     0.764      4000
weighted avg      0.771     0.765     0.764      4000

[틀린 문장 5개]
  정답 긍정 / 예측 부정 | 아슈반 옛날생각나
  정답 긍정 / 예측 부정 | 아뭘자꾸공백을없애래10자썼잖아OO
  정답 부정 / 예측 긍정 | 특수효과 장면이 암울합니다...한국영화 기술의 암울했던 과거
  정답 부정 / 예측 긍정 | 생활환경이 비슷해야 공감할 수 있다.
  정답 긍정 / 예측 부정 | 역시 오우삼^^
```

> **관찰 ★**: 틀린 문장을 읽어 보세요. *'아슈반 옛날생각나'*, *'역시 오우삼^^'* 처럼 **감정을 나타내는 단어가 없는 문장**, 띄어쓰기가 없어 긴 덩어리(`<unk>`)가 되는 문장, *'…암울했던 과거'* 처럼 맥락을 알아야 하는 문장을 틀립니다. 재현율은 부정 0.840 / 긍정 0.692 — **긍정 리뷰를 부정으로 판단하는 실수가 더 많습니다.**

---

## 6단계 — 내 문장 넣어 보기

아래 `MY_SENTENCES` 에 **직접 쓴 리뷰**를 추가해 보세요. 세 번째 문장처럼 **앞은 칭찬, 끝은 혹평**인 문장을 꼭 시험해 봅니다.

In [ ]:
# 셀 7 — 문장 하나의 긍정/부정 확률
@torch.no_grad()
def predict(text):
    x = torch.tensor([encode(text, vocab, MAX_LEN)]).to(device)    # 배치 차원을 붙여 (1, 40)
    model.eval()
    # 빈칸 ⑩ (1줄) — 힌트 : 점수 2개를 확률로 바꾼다.  torch.softmax(model(x), dim=1)[0]
    p = ____
    return p[0].item(), p[1].item()

MY_SENTENCES = [
    "정말 재미있었다 또 보고 싶다",
    "시간이 아까웠다",
    "배우도 좋고 영상도 훌륭한데 결말이 최악이다",
    "기대 안 했는데 생각보다 괜찮았다",
]
for s in MY_SENTENCES:
    neg, pos = predict(s)
    print(f"{'긍정' if pos > neg else '부정'}  (긍정 {pos*100:5.1f}%)  {s}")

**기대 출력**
```
긍정  (긍정  96.5%)  정말 재미있었다 또 보고 싶다
부정  (긍정   7.8%)  시간이 아까웠다
긍정  (긍정  72.1%)  배우도 좋고 영상도 훌륭한데 결말이 최악이다
부정  (긍정  25.3%)  기대 안 했는데 생각보다 괜찮았다
```

> **생각해 볼 것**: 세 번째 문장 *'배우도 좋고 영상도 훌륭한데 결말이 최악이다'* 를 **긍정 72.1%** 로 틀렸습니다. 앞의 칭찬이 요약을 긍정 쪽으로 밀어 놓은 데다(2교시 장기 의존성), '훌륭한데' 는 훈련 문장에 한 번도 없어 `<unk>` 가 됐고 '최악이다' 는 32번뿐입니다(같은 뜻의 '최악' 72번, '최악의' 105번이 따로 셉니다 — 1교시 토큰의 한계). 네 번째 문장은 '안'(훈련 문장 132번 중 긍정 30%) 쪽으로 기울어 부정이 됐습니다.

In [ ]:
# 셀 8 — 결과 저장  (빈칸 없음)  ★ 어휘사전은 모델과 반드시 함께 저장한다
save_vocab(vocab, "models/vocab.json")
json.dump({"모델": "LSTM", "검증정확도": round(lstm_hist["best_acc"], 4),
           "파라미터": sum(p.numel() for p in model.parameters()), "학습시간초": round(lstm_hist["time"], 1)},
          open("results/lstm_result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("저장 : models/lstm_sentiment.pt · models/vocab.json · results/lstm_result.json")

> **왜 어휘사전도 저장하나**: 모델은 *"842번 단어"* 만 압니다. 842번이 어떤 단어인지는 어휘사전에만 있습니다.
> 어휘사전 없이 모델만 저장하면, 다음에 불러와도 **문장을 번호로 바꿀 수 없습니다.** 11주차의 토크나이저도 같은 이유로 함께 저장합니다.

---

## 도전 (선택) — 어텐션: 모델이 어느 단어를 봤나

2교시의 질문 *"마지막 요약 하나에 문장 전체를 담는 게 맞나?"* 에 대한 답입니다.
**모든 시점의 요약을 남겨 두고, 단어마다 중요도(가중치)를 매겨 섞습니다.**

```
  단어     배우도  좋고  영상도  훌륭한데  결말이  최악이다
  가중치    0.02   0.03   0.01    0.04     0.10    0.80     ← 합 = 1
            └──────────── 가중치대로 섞는다 ─────────────→ 분류
```

빈칸은 없습니다. 실행하고 **가중치 그림**을 읽어 보세요.

In [ ]:
# 셀 9 — (선택) LSTM + 어텐션
class LSTMAttention(LSTMClassifier):
    def __init__(self, vocab_size, embed=128, hidden=128):
        super().__init__(vocab_size, embed, hidden)
        self.score = nn.Linear(hidden, 1)                  # 시점마다 중요도 점수 하나

    def forward(self, x, return_weights=False):
        out, _ = self.lstm(self.emb(x))                    # (B, 40, 128)  모든 시점을 남긴다
        s = self.score(out).squeeze(-1)                    # ① 점수  (B, 40)
        s = s.masked_fill(x == 0, -1e9)                    #    패딩 자리는 아주 작게 → 가중치 0
        w = torch.softmax(s, dim=1)                        # ② 합이 1 인 가중치
        ctx = (out * w.unsqueeze(-1)).sum(dim=1)           # ③ 가중치대로 섞는다  (B, 128)
        logits = self.fc(self.drop(ctx))
        return (logits, w) if return_weights else logits

set_seed()
attn_model = LSTMAttention(VOCAB_SIZE).to(device)
attn_hist = fit(attn_model, epochs=EPOCHS, lr=1e-3, save_path="models/lstm_attention.pt")
attn_model.load_state_dict(torch.load("models/lstm_attention.pt"))
print(f"\nLSTM {lstm_hist['best_acc']*100:.2f}%  vs  LSTM+어텐션 {attn_hist['best_acc']*100:.2f}%")

**기대 출력** (끝부분)
```
epoch 5 | 훈련 손실 0.2192 | 검증 손실 0.5955 | 검증 정확도 76.55%
최고 검증 정확도 77.08%  |  학습 시간 456.6초  |  저장 : models/lstm_attention.pt
LSTM 76.55%  vs  LSTM+어텐션 77.08%
```

In [ ]:
# 셀 10 — (선택) 어텐션 가중치 그리기
@torch.no_grad()
def show_attention(text, ax):
    ids = encode(text, vocab, MAX_LEN)
    n = sum(1 for i in ids if i != 0)
    attn_model.eval()
    logits, w = attn_model(torch.tensor([ids]).to(device), return_weights=True)
    pos = torch.softmax(logits, 1)[0, 1].item()
    words = decode(ids, vocab)
    ax.bar(range(n), w[0, :n].cpu(), color="tab:orange")
    ax.set_xticks(range(n)); ax.set_xticklabels(words, rotation=30, fontsize=11)
    ax.set_title(f"{text}  →  긍정 {pos*100:.0f}%", fontsize=12); ax.set_ylim(0, 1)

SAMPLES = ["배우도 좋고 영상도 훌륭한데 결말이 최악이다",
           "기대 안 했는데 생각보다 괜찮았다",
           "스토리가 지루하고 연기도 어색하다"]
fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(9, 2.6 * len(SAMPLES)))
for s, ax in zip(SAMPLES, axes):
    show_attention(s, ax)
plt.tight_layout(); plt.savefig("results/attention.png", dpi=120); plt.show()

> **관찰 ★**: 세 번째 문장은 '지루하고'·'연기도'·'어색하다' 에 비중이 고르게 실려 부정(긍정 1%)으로 맞혔습니다. 그런데 첫 문장은 **어텐션 모델도 긍정 95% 로 틀렸고**, 비중이 가장 큰 단어는 '영상도'(약 0.35)였습니다. '최악이다' 는 훈련 문장에 32번뿐이라 강한 신호가 되지 못했습니다. **어텐션은 정답을 보장하지 않지만, 왜 틀렸는지 들여다볼 수 있게 해 줍니다.** 정확도는 LSTM 76.55% vs 어텐션 77.08% 로 차이가 작습니다.
> 9주차에 *"혼동행렬로 무엇을 헷갈렸는지 본다"* 였다면, 어텐션은 *"문장의 어느 단어를 보고 판단했는지"* 를 보여 줍니다.

---

## 과제 — LSTM 감성 분류 결과 보고

| 제출물 | 내용 |
|---|---|
| ① 노트북 | 이 노트북을 **`15_lstm_sentiment.ipynb`** 로 저장 (빈칸 10곳 완성 · 실행 결과 포함) |
| ② 학습 곡선 해석 | 과적합 신호가 보였는가? **몇 번째 epoch 이 가장 좋았나?** 근거 2줄 |
| ③ 틀린 문장 분석 | 셀 6 의 틀린 문장 중 **2개**를 골라, 모델이 왜 틀렸을지 추측 2줄씩 |
| ④ 내 문장 | 직접 쓴 리뷰 **3개**의 예측 결과 (맞은 것·틀린 것 모두 괜찮음) |
| ⑤ (선택) 어텐션 | 도전 셀을 했다면 가중치 그림 1장과 *"모델이 주목한 단어가 납득되는가"* 1줄 |

> **가장 중요한 것은 ③ 입니다.** 정확도 숫자보다 *"모델이 어떤 문장을 어려워하는가"* 를 읽어 내는 것이 목표입니다.

### 체크리스트

- [ ] 어휘사전을 **훈련 문장으로만** 만들었다
- [ ] 문장이 번호 리스트(길이 40)로 바뀌는 것을 확인했다
- [ ] `Embedding → LSTM → Linear` 모델을 만들고 출력 shape `(64, 2)` 를 확인했다
- [ ] 학습 루프를 직접 쓰고, 검증 정확도가 가장 좋을 때 저장했다
- [ ] 혼동행렬과 틀린 문장을 보고 모델의 약점을 말할 수 있다
- [ ] 내 문장으로 예측해 봤다
- [ ] 어휘사전을 모델과 함께 저장했다